# Record DeepGlobe Demo Video (Minimal)

Self-contained notebook that renders a **deterministic MP4** of
DeepLabV3+-R50 predictions on 3-5 satellite images. Output is a
1920&times;1080 video you can embed in your slides as a backup to the
live Gradio demo.

**No Gradio here**, so the dependency surface is small and nothing
will drag numpy down to 1.x. Just `segmentation-models-pytorch`.

**Prerequisite**: trained checkpoint + raw DeepGlobe images on your Google Drive.

**Runtime**: `Runtime -> Change runtime type -> GPU (T4)` &mdash; one 2448&sup2; image takes ~1 s on a T4.

## 1. Clone repo and install dependencies

Only `segmentation-models-pytorch>=0.4` is installed. Colab already
ships compatible `numpy`, `opencv-python`, `torch`, `matplotlib`, `tqdm`.

In [ ]:
%cd /content
![ -d landcover-seg ] || git clone https://github.com/Jerryzhang258/landcover-seg
%cd /content/landcover-seg
!git pull --quiet origin main || true

# Single dependency. No -q so pip errors are visible.
!pip install -U "segmentation-models-pytorch>=0.4"

# Per-package verification — if any of these raises, the install failed.
!python -c "import numpy; assert numpy.__version__.startswith('2'), f'numpy got downgraded to {numpy.__version__} — Disconnect runtime and re-run'; print('OK numpy ', numpy.__version__)"
!python -c "import cv2;   print('OK cv2   ', cv2.__version__)"
!python -c "import torch; print('OK torch ', torch.__version__, 'cuda=' + str(torch.cuda.is_available()))"
!python -c "import segmentation_models_pytorch as smp; print('OK smp   ', smp.__version__)"

## 2. Mount Google Drive and set paths

Edit `CKPT_PATH`, `SRC_IMG_DIR`, and the three image IDs in `EXAMPLE_IDS`.
Pick images that showcase the paper's findings: one mixed urban-rural,
one barren-heavy, one rangeland-heavy (the known failure mode).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

# EDIT ME: path to your trained checkpoint on Drive.
CKPT_PATH   = '/content/drive/MyDrive/landcover/checkpoints/deeplab_r50_combined_best.pt'
MODEL_NAME  = 'deeplab_r50'

# EDIT ME: source directory of raw DeepGlobe images on Drive + IDs to include.
SRC_IMG_DIR = '/content/drive/MyDrive/landcover/data/raw/train'
EXAMPLE_IDS = ['104876', '153214', '629571']  # replace with your choices

VIDEO_OUT   = 'demo/demo_video.mp4'
POSTER_OUT  = 'demo/demo_poster.png'
SECONDS_EACH = 5.0

# Validate everything before running the expensive inference step.
assert os.path.isfile(CKPT_PATH), f'checkpoint not found: {CKPT_PATH}'
VIDEO_IMAGES = [f'{SRC_IMG_DIR}/{iid}_sat.jpg' for iid in EXAMPLE_IDS]
missing = [p for p in VIDEO_IMAGES if not os.path.isfile(p)]
assert not missing, f'these images were not found on Drive: {missing}'

print(f'ckpt:  {CKPT_PATH}')
print(f'images ({len(VIDEO_IMAGES)}):')
for p in VIDEO_IMAGES:
    print(f'  {p}')

## 3. Render the MP4

Expected time on a T4 GPU: ~1 s of inference per image + ~5 s of rendering
per image. For 3 images at 5 s hold time, the final video is 15 s long.

In [ ]:
img_args = ' '.join(f'"{p}"' for p in VIDEO_IMAGES)
!python demo/record_video.py \
    --ckpt "$CKPT_PATH" \
    --model "$MODEL_NAME" \
    --images $img_args \
    --out "$VIDEO_OUT" \
    --poster "$POSTER_OUT" \
    --seconds-per-image $SECONDS_EACH

## 4. Preview inline + copy to Drive

In [ ]:
from IPython.display import HTML
import base64
with open(VIDEO_OUT, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
HTML(f'<video width="960" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

In [ ]:
# Copy the MP4 and poster back to Drive so you can grab them on your laptop.
import shutil
DRIVE_OUT_DIR = '/content/drive/MyDrive/landcover/demo'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
shutil.copy(VIDEO_OUT, DRIVE_OUT_DIR)
shutil.copy(POSTER_OUT, DRIVE_OUT_DIR)
print(f'copied to {DRIVE_OUT_DIR}:')
!ls -lh "$DRIVE_OUT_DIR"